# 🔒 RAG local para documentos privados con LangChain + Ollama

![Python](https://img.shields.io/badge/Python-3.12-blue)
![Ollama](https://img.shields.io/badge/Ollama-local-black)
![ChromaDB](https://img.shields.io/badge/ChromaDB-vector%20store-orange)

## 📖 Sobre este notebook

Adaptación **100% local** del lab de IBM / Skills Network
*"Summarize Private Documents Using RAG, LangChain, and LLMs"*. El lab original
corre en el JupyterLab web de Skills Network y consume el LLM desde **watsonx.ai**
con un `project_id="skills-network"` que solo funciona dentro de ese entorno.
Como el kernel web no arrancó, se rehízo el recorrido completo en local con:

- **[Ollama](https://ollama.com/)** como servidor de modelos (sin API keys, sin costo).
- **`llama3.2`** (3B) y **`qwen2.5:7b`** como LLM, para poder compararlos.
- **`nomic-embed-text`** como modelo de embeddings (reemplaza a `HuggingFaceEmbeddings`).
- **ChromaDB** como vector store y **LangChain** para orquestar todo.

**Por qué local:** el escenario del lab es un empleado nuevo con documentos
*privados* que no puede subir a un servicio público. Con Ollama, el LLM y los
embeddings corren en tu equipo, así que con documentos reales nada saldría de él.
(Los dos `.txt` de ejemplo sí se descargan desde el bucket público del curso.)

### 🎯 Objetivos (los mismos del lab original)
1. Dividir y convertir documentos en embeddings que un LLM pueda aprovechar.
2. Usar distintos LLM y elegir el más adecuado (aquí: modelos locales en vez de watsonx).
3. Implementar cadenas de recuperación de LangChain para distintos propósitos.
4. Construir un agente con RAG y **memoria de conversación**.

## ✅ Requisitos previos

1. **[Ollama](https://ollama.com/download)** instalado y corriendo.
2. Descargar los modelos usados:
   ```bash
   ollama pull llama3.2
   ollama pull qwen2.5:7b
   ollama pull nomic-embed-text
   ```
3. **Python 3.12** y un entorno virtual (se prefirió 3.12 sobre 3.14 por compatibilidad de dependencias, sin llegar a probar 3.14).
4. Instalar dependencias (ver `requirements.txt` en esta carpeta):
   ```bash
   pip install -r requirements.txt
   ```
5. **En VS Code:** seleccionar como kernel el intérprete del `venv` (ver Sección 1).

## 🗂️ Contenido

1. Verificación del entorno
2. Indexing: cargar, dividir, embeber y almacenar
3. Retrieval y generación: primer RAG
4. Alucinaciones: prompt estricto y verificación contra el texto
5. Cambio de modelo: `llama3.2` → `qwen2.5:7b`
6. Memoria conversacional
7. Agente interactivo
8. Ejercicios (2.º documento, fuentes, comparación de modelos)
9. Conclusiones

## 🧭 Cómo leer este notebook

- Las **salidas guardadas** son de una ejecución real: puedes leer los resultados en GitHub sin correr nada.
- `llm` y `PROMPT` se **reasignan** varias veces; el notebook depende del orden de ejecución.

| Celdas | `llm` activo | Temperatura | Prompt de RAG |
|---|---|---|---|
| 6 – 11 | `llama3.2` | 0.5 | ninguno (7–9) y "no sé" del lab (10–11) |
| 12 – 14 | `llama3.2` | 0 | estricto (definido en la celda 12) |
| 15 en adelante | `qwen2.5:7b` | 0 | estricto |

- Las celdas con `temperature=0.5` (6–11) pueden dar respuestas distintas si las vuelves a ejecutar.
- ⚠️ **Antes de re-ejecutar todo, reinicia el kernel** (*Restart & Run All*): `Chroma.from_documents` vuelve a insertar los chunks en la misma colección y quedan **duplicados** (verificado: 3 → 6 al ejecutarlo dos veces).

## ⚠️ Nota honesta sobre modelos pequeños locales

Buena parte del valor de este lab es **ver en vivo** cómo falla un modelo de 3B
(inventa, responde sobre un tema cercano) y aprender a diagnosticarlo con
evidencia, en lugar de asumir que "el código está mal".

---

## 1. Verificación del entorno

### Celda 1: ¿Qué Python usa el kernel?

El primer problema real de este lab fue un `ModuleNotFoundError: No module named 'wget'`
en un entorno donde `wget` **sí** estaba instalado. La causa: el kernel de VS Code
apuntaba a **Miniconda** (y luego a un Python 3.12 global), no al `venv` del proyecto.

Esta celda imprime el intérprete real del kernel. La ruta debe terminar en
`...\venv\Scripts\python.exe` (en macOS/Linux, `.../venv/bin/python`). Si no, hay que
cambiar el kernel (*Select Kernel → Python Environments → venv*).

In [1]:
# ==============================================================================
# CELDA 1: Verificar qué intérprete de Python usa el kernel
# ==============================================================================
import sys

# La ruta debe apuntar al python DENTRO del venv del proyecto.
# Si apunta a Miniconda o a un Python global, las librerías instaladas en el venv
# no se encontrarán y aparecerá un ModuleNotFoundError.
print(sys.executable)

<proyecto>\venv\Scripts\python.exe


### Celda 2: Imports

Se agrupan todos los imports en un solo lugar. Diferencias respecto al lab original:
`OllamaEmbeddings` y `OllamaLLM` reemplazan a `HuggingFaceEmbeddings` y a
`WatsonxLLM`/`ibm_watsonx_ai`. El paquete `langchain-classic` agrupa las *chains* y la
memoria "clásicas" (`RetrievalQA`, `ConversationalRetrievalChain`,
`ConversationBufferMemory`) que usa este lab.

In [2]:
# ==============================================================================
# CELDA 2: Imports
# ==============================================================================
import os                               # comprobar si un archivo ya existe antes de descargarlo
import warnings
warnings.filterwarnings("ignore")       # intenta silenciar warnings (varios avisos de LangChain se muestran igual: ver Celda 17)

import wget                             # descarga los .txt de ejemplo

# --- LangChain: carga, división y almacenamiento de documentos ---
from langchain_community.document_loaders import TextLoader     # lee un .txt como Document
from langchain_text_splitters import CharacterTextSplitter      # divide el texto en chunks
from langchain_chroma import Chroma                             # vector store (paquete dedicado)

# --- LangChain "classic": chains de RAG y memoria ---
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# --- Ollama: modelos locales (reemplaza a HuggingFace e IBM watsonx del lab original) ---
from langchain_ollama import OllamaEmbeddings, OllamaLLM

print("All imports successful!")

All imports successful!


## 2. Indexing: cargar, dividir, embeber y almacenar

Un sistema RAG tiene dos fases. La primera (**Indexing**) se hace una vez por documento:

```text
INDEXING (una vez)
  Load ──► Split ──► Embed ──► Store
  TextLoader   CharacterTextSplitter   OllamaEmbeddings   Chroma

RETRIEVAL + GENERATION (en cada pregunta)
  Pregunta ──► Retrieve (top-k chunks) ──► Prompt + contexto ──► LLM ──► Respuesta
               retriever                   PromptTemplate         OllamaLLM
```

### Celda 3: Descargar el documento de ejemplo

`companyPolicies.txt` describe nueve políticas de una empresa ficticia (fumar,
móviles, alcohol y drogas, etc.).

⚠️ **Detalle real:** `wget.download` **no sobrescribe** un archivo existente; crea
`companyPolicies (1).txt`. Por eso se comprueba antes con `os.path.exists`
(cambio respecto al lab original).

In [3]:
# ==============================================================================
# CELDA 3: Descargar el documento (solo si no existe)
# ==============================================================================
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# wget.download crea "companyPolicies (1).txt" si el archivo ya existe,
# en vez de sobrescribirlo. Se comprueba antes para no acumular copias.
if not os.path.exists(filename):
    wget.download(url, out=filename)
    print('file downloaded')
else:
    print('file already exists, skipping download')

file downloaded


### Celda 4: Cargar y dividir en chunks

Un LLM tiene una ventana de contexto limitada y buscar sobre un documento entero es
poco preciso, así que el texto se corta en trozos (*chunks*).

`CharacterTextSplitter` **solo corta donde encuentra su separador** (`"\n\n"` por
defecto). Por eso `chunk_size=1000` es un *objetivo*, no un límite: si entre dos
saltos de línea dobles hay más de 1000 caracteres, el chunk queda más grande y
LangChain avisa con `Created a chunk of size ... which is longer than the specified 1000`.

In [4]:
# ==============================================================================
# CELDA 4: Cargar el documento y dividirlo en chunks
# ==============================================================================
# encoding="utf-8" explícito: el valor por defecto depende de la configuración regional
# del sistema (en Windows puede no ser UTF-8).
loader = TextLoader(filename, encoding="utf-8")
documents = loader.load()          # lista con 1 Document: el archivo completo

# Solo corta en "\n\n". chunk_overlap=0 -> los chunks no comparten texto entre sí.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(len(texts))                  # cantidad de chunks resultantes

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


**Resultado observado:** 16 chunks, y **7 de ellos superan los 1000 caracteres**
(entre 1624 y 2032). Cada política del documento es un bloque sin líneas en blanco
internas, así que el splitter no tiene dónde cortarla. Es la primera pista de que
*la estructura del texto manda más que `chunk_size`* (se retoma en la Celda 32 y en
las conclusiones).

### Celda 5: Embeddings y almacenamiento en Chroma

Un modelo de **embeddings** convierte cada chunk en un vector numérico; textos con
significado parecido quedan con vectores cercanos. **Chroma** guarda esos vectores y
permite buscar los más parecidos a una pregunta.

- Se usa `nomic-embed-text` (vía Ollama) en lugar de `HuggingFaceEmbeddings()` del lab
  original: evita instalar PyTorch y `sentence-transformers`. Al ser otro modelo de
  embeddings, el ranking de chunks **no será idéntico** al del lab de IBM.
- Sin `persist_directory`, Chroma vive **en memoria** (se pierde al cerrar el kernel).
- ⚠️ Ejecutar esta celda dos veces en el mismo kernel **duplica** los chunks.

In [5]:
# ==============================================================================
# CELDA 5: Embeddings locales + vector store (Chroma)
# ==============================================================================
embeddings = OllamaEmbeddings(model="nomic-embed-text")   # requiere: ollama pull nomic-embed-text

# Embebe cada chunk y lo guarda en Chroma (en memoria, colección por defecto).
# OJO: re-ejecutar esta celda sin reiniciar el kernel duplica los chunks.
docsearch = Chroma.from_documents(texts, embeddings)
print('document ingested')

document ingested


## 3. Retrieval y generación: primer RAG

### Celda 6: LLM local y prueba **sin** contexto

Antes de usar RAG conviene ver qué pasa **sin** él. `OllamaLLM` es el wrapper de
LangChain hacia el servidor local de Ollama.

In [6]:
# ==============================================================================
# CELDA 6: LLM local (llama3.2) y prueba directa, sin RAG
# ==============================================================================
llm = OllamaLLM(
    model="llama3.2",    # llama3.2 = modelo de 3B parámetros
    temperature=0.5,     # 0 = casi determinista; valores altos = más variedad
    num_predict=256,     # máximo de tokens que puede generar por respuesta
)

# Pregunta directa al modelo, sin pasarle ningún documento.
print(llm.invoke("Responde en una frase: ¿qué es RAG?"))

RAG puede referirse a varias cosas, pero algunas posibles interpretaciones incluyen "Red de Ayuda y Rescate" o "Reacción Antigeno de Glóbulos", dependiendo del contexto.


**Resultado observado:** la respuesta es **incorrecta**: interpreta "RAG" como una
"Red de Ayuda y Rescate" o una "Reacción Antigeno de Glóbulos", sin relación con
*Retrieval-Augmented Generation*. Sin contexto, el modelo responde de memoria y con
seguridad aunque no sepa. Ese es exactamente el problema que RAG intenta reducir.

### Celda 7: Primera cadena RAG (`RetrievalQA`) con una pregunta puntual

`RetrievalQA` une el retriever y el LLM: busca los chunks más parecidos a la
pregunta, los inserta en el prompt y le pide al modelo que responda.

- `chain_type="stuff"`: mete **todos** los chunks recuperados en un único prompt.
- `as_retriever()`: por defecto devuelve los **k = 4** chunks más similares.

In [7]:
# ==============================================================================
# CELDA 7: RetrievalQA - pregunta puntual
# ==============================================================================
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",                    # "stuff" = todos los chunks recuperados en UN solo prompt
    retriever=docsearch.as_retriever(),    # por defecto trae los k=4 chunks más similares a la pregunta
    return_source_documents=False,         # True devolvería también los chunks usados (ver Celda 31)
)

query = "what is mobile policy?"
print(qa.invoke(query)["result"])          # invoke devuelve un dict; la respuesta está en "result"

The Mobile Phone Policy is a set of guidelines that outlines standards and expectations for the use of mobile devices in the organization, emphasizing responsible and secure usage to ensure compliance with company values and legal requirements.


**Resultado observado:** una descripción coherente de la política de móviles. Con
preguntas **puntuales**, donde existe un chunk que contiene la respuesta, RAG funciona bien.

### Celda 8: Pregunta de alto nivel (resumen)

In [8]:
# ==============================================================================
# CELDA 8: Pregunta amplia - "resume el documento"
# ==============================================================================
query = "Can you summarize the document for me?"
print(qa.invoke(query)["result"])

The document appears to be an employee handbook or code of conduct for a company. It outlines various policies related to smoking, workplace behavior, performance expectations, disciplinary actions, termination procedures, and other aspects of employment.

The policies cover topics such as:

* Smoking restrictions and designated areas
* Compliance with applicable laws and regulations
* Proper disposal of smoking materials
* Prohibition on smoking in company vehicles and enclosed spaces
* Disciplinary actions for non-compliance
* Termination procedures and fairness
* Exit processes for departing employees

Overall, the document aims to establish clear guidelines and expectations for employees, contractors, and temporary staff to maintain a productive, ethical, and respectful work environment.


**Resultado observado:** un resumen plausible pero **parcial por diseño**: el modelo
solo ve los 4 chunks (de 16) más parecidos a la frase *"summarize the document"*, no el
documento entero. Es consistente con el hallazgo del lab 02 del repo: la búsqueda por
similitud rinde en preguntas puntuales y flojea en resúmenes globales.

### Celda 9: Pregunta trampa (la respuesta **no** está en el documento)

El documento habla de fumar en vehículos de la empresa, pero **no menciona comida**.
Esta pregunta sirve para medir si el modelo admite que no sabe o inventa.

In [9]:
# ==============================================================================
# CELDA 9: Pregunta trampa - "¿puedo comer en los vehículos de la empresa?"
# ==============================================================================
# El documento NO habla de comer. Lo correcto sería admitir que no lo menciona.
query = "Can I eat in company vehicles?"
print(qa.invoke(query)["result"])

According to the Smoking Policy, smoking is not permitted in company vehicles, whether they are owned or leased, to maintain the condition and cleanliness of these vehicles.


**Resultado observado:** el modelo responde sobre **fumar**. El retriever trajo la
política de fumar porque menciona *vehicles*, y el modelo contestó un tema cercano en
vez de admitir que el documento no habla de comida.

## 4. Alucinaciones: prompt estricto y verificación contra el texto

### Celda 10: Prompt template del lab

Un `PromptTemplate` permite darle instrucciones al modelo. `{context}` y `{question}`
son las dos variables que `RetrievalQA` rellena automáticamente (chunks recuperados y
pregunta). Aquí se le pide decir "no sé" en lugar de inventar.

In [10]:
# ==============================================================================
# CELDA 10: Prompt template (versión del lab): "si no sabes, dilo"
# ==============================================================================
# {context}  -> RetrievalQA lo rellena con los chunks recuperados
# {question} -> RetrievalQA lo rellena con la pregunta del usuario
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definitely do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}   # se pasa a la chain para reemplazar su prompt por defecto

### Celda 11: La misma pregunta trampa, ahora con el prompt

In [11]:
# ==============================================================================
# CELDA 11: Pregunta trampa con el prompt "no sé"
# ==============================================================================
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs,     # <- aquí entra el prompt personalizado
    return_source_documents=False,
)

query = "Can I eat in company vehicles?"
print(qa.invoke(query)["result"])

No, according to the Smoking Policy, smoking is not permitted in company vehicles, whether they are owned or leased, to maintain the condition and cleanliness of these vehicles.


**Resultado observado:** el prompt **no fue suficiente** con `llama3.2`. Responde con un
"No" seguido de la política de fumar, es decir, sigue contestando un tema que no se preguntó.

### Celda 12: Prompt estricto + `temperature=0`

Se endurece el prompt (frase de rechazo exacta, prohibido inferir) y se baja la
temperatura a 0 para que la salida sea lo más determinista posible. Se activa
`return_source_documents=True` para poder inspeccionar los chunks (Ejercicio 2).

In [12]:
# ==============================================================================
# CELDA 12: Prompt estricto + temperatura 0
# ==============================================================================
llm = OllamaLLM(model="llama3.2", temperature=0, num_predict=256)   # temperature=0: casi determinista

# Prompt más restrictivo: frase exacta de rechazo y prohibición explícita de inferir.
prompt_template = """You are answering questions using ONLY the context below.

Rules:
- Answer only if the context explicitly addresses the exact topic of the question.
- If the context does not directly mention the topic asked (even if it mentions related topics), reply exactly: "The document does not mention this."
- Do not infer, guess, or answer about a different topic.

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True             # devuelve también los chunks usados
)

query = "Can I eat in company vehicles?"
result = qa.invoke(query)
print(result["result"])

No, according to the context, smoking (including eating) is not permitted in company vehicles.


**Resultado observado:** ni el prompt estricto ni `temperature=0` bastaron. El modelo
afirma que *"smoking (including eating) is not permitted in company vehicles"*: la
parte de **"including eating" no aparece en el documento**, es una invención.
Las Celdas 13 y 14 lo comprueban con el texto original.

### Celda 13: Verificación ingenua (falsa alarma) ⚠️

Para comprobar si el documento habla de comida se probó primero con el operador `in`.
Se deja en el notebook a propósito, como lección: el resultado es engañoso.

In [13]:
# ==============================================================================
# CELDA 13: Verificación INGENUA (da un falso positivo; se deja como lección)
# ==============================================================================
texto = documents[0].page_content.lower()

# "eat" in texto busca la SUBCADENA "eat" dentro de cualquier palabra.
# Devuelve True aunque el documento nunca hable de comer.
print("¿aparece 'eat' o 'food'?:", "eat" in texto or "food" in texto)
print("¿aparece 'smoking'?:", "smoking" in texto)

¿aparece 'eat' o 'food'?: True
¿aparece 'smoking'?: True


### Celda 14: Verificación correcta con palabras completas (`\b`)

La expresión regular `\bpalabra\b` exige que sea una **palabra completa**, y además se
imprime el contexto de cada coincidencia.

In [14]:
# ==============================================================================
# CELDA 14: Verificación CORRECTA - palabras completas con regex
# ==============================================================================
import re

texto = documents[0].page_content

# \b = límite de palabra: "eat" NO coincide dentro de "create" o "great".
for palabra in ["eat", "eating", "food", "drink", "meal"]:
    coincidencias = [m.start() for m in re.finditer(rf"\b{palabra}\b", texto, re.IGNORECASE)]
    print(f"'{palabra}': {len(coincidencias)} coincidencias")
    for pos in coincidencias[:3]:   # muestra hasta 3 fragmentos de contexto por palabra
        print("   ...", texto[max(0, pos-80):pos+80].replace("\n", " "), "...")

'eat': 0 coincidencias
'eating': 0 coincidencias
'food': 0 coincidencias
'drink': 0 coincidencias
'meal': 0 coincidencias


**Resultado observado:** 0 coincidencias para las cinco palabras. Queda demostrado que el
documento **no habla de comer ni de comida** y que la respuesta del modelo era una
alucinación. La Celda 13 dio `True` porque la subcadena `eat` aparece dentro de otras
palabras. **Lección:** hay que verificar también el método de verificación.

## 5. Cambio de modelo: `llama3.2` → `qwen2.5:7b`

### Celda 15: Misma prueba, otro modelo

Como el prompt no bastó, se cambia el modelo (7B en vez de 3B) dejando **todo lo demás
igual**: mismo retriever, mismo prompt estricto, `temperature=0`, mismas preguntas. Así,
la diferencia se debe al modelo. Se añade una segunda pregunta (con respuesta real en el
documento) para confirmar que el modelo no se limita a negarse siempre.

La variable `MODELO` permite comparar otros modelos cambiando una sola línea.

In [15]:
# ==============================================================================
# CELDA 15: Mismo prompt estricto, modelo más grande (qwen2.5:7b)
# ==============================================================================
MODELO = "qwen2.5:7b"   # cambia aquí para comparar otros modelos (requiere: ollama pull <modelo>)

llm = OllamaLLM(model=MODELO, temperature=0, num_predict=256)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs={"prompt": PROMPT},   # el prompt estricto de la Celda 12
    return_source_documents=True
)

# Pregunta 1: trampa (no está en el documento). Pregunta 2: control (sí está).
for q in ["Can I eat in company vehicles?", "what is mobile policy?"]:
    print("Q:", q)
    print("A:", qa.invoke(q)["result"], "\n")

Q: Can I eat in company vehicles?
A: The document does not mention this. 

Q: what is mobile policy?
A: The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. It ensures that employees utilize mobile phones in a manner consistent with company values and legal compliance. Key aspects include acceptable use, security measures, confidentiality guidelines, cost management, compliance with laws and regulations, handling lost or stolen devices, and potential consequences for non-compliance. 



**Resultado observado — pregunta trampa `"Can I eat in company vehicles?"`:**

| Config. | Modelo | Prompt | Temp. | Respuesta |
|---|---|---|---|---|
| A (celda 9) | `llama3.2` | por defecto | 0.5 | Responde sobre **fumar** |
| B (celda 11) | `llama3.2` | "no sé" del lab | 0.5 | "No, ... smoking is not permitted..." |
| C (celda 12) | `llama3.2` | estricto | 0 | "smoking **(including eating)** is not permitted" → **inventa** |
| D (celda 15) | `qwen2.5:7b` | estricto | 0 | "The document does not mention this." ✅ |

Con la pregunta de control, `qwen2.5:7b` respondió con contenido sobre la política de
móviles, así que no se trata de un modelo que se niega a todo.

> ⚠️ **Cautela:** es *una* pregunta trampa y *una* corrida por configuración, y cambian a la
> vez el tamaño (3B → 7B) y la familia (Llama → Qwen). No es un benchmark: sirve para
> ver el fenómeno, no para afirmar que un modelo es mejor en general.

## 6. Memoria conversacional

### Celda 16: El problema sin memoria

Cada pregunta se procesa de forma aislada. Si se pregunta *"What can I not do in it?"*
sin decir a qué se refiere "it", el modelo no puede saberlo.

In [16]:
# ==============================================================================
# CELDA 16: Pregunta con pronombre ambiguo, SIN memoria
# ==============================================================================
# "it" no significa nada aquí: esta chain no recuerda la pregunta anterior.
print(qa.invoke("What can I not do in it?")["result"])

The document does not mention this.


**Resultado observado:** `"The document does not mention this."`. Con el prompt estricto,
el fallo es *seguro* (no inventa), pero tampoco puede responder.

### Celda 17: Cadena con memoria (`ConversationalRetrievalChain`)

`ConversationalRetrievalChain` añade un paso previo: usa el historial para **reescribir**
la pregunta de seguimiento como una pregunta independiente ("What is the aim of
*the smoking policy*?") y solo entonces busca en el documento. La memoria
(`ConversationBufferMemory`) guarda ese historial.

- Se usa `return_messages=True` (el lab original escribe `return_message`, sin la *s*,
  que no es el nombre del parámetro).
- Ya no hace falta pasar el historial a mano: la memoria lo gestiona.
- ⚠️ La salida de esta celda son **avisos de deprecación**: `ConversationBufferMemory` está
  marcada como obsoleta (retirada prevista en LangChain 2.0). Funciona hoy, pero es un
  patrón *legacy*; el propio aviso recomienda usar `create_agent` con *checkpointing*
  (o la API `Store`) para conservar el historial.
- Esta chain usa los prompts **por defecto** de LangChain, no el prompt estricto.

In [17]:
# ==============================================================================
# CELDA 17: Cadena conversacional con memoria
# ==============================================================================
memory = ConversationBufferMemory(
    memory_key="chat_history",   # nombre con el que la chain busca el historial
    return_messages=True,        # guarda objetos de mensaje (Human/AI), no un string plano
    output_key="answer"          # necesario si se activa return_source_documents
)

qa_mem = ConversationalRetrievalChain.from_llm(
    llm=llm,                                  # aquí llm = qwen2.5:7b
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    memory=memory,                            # la memoria se actualiza sola en cada invoke
    return_source_documents=False
)

<celda>:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
<proyecto>\venv\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


### Celda 18: Conversación con preguntas de seguimiento

Se hacen tres preguntas encadenadas. En la 2.ª y la 3.ª, "it" solo se entiende gracias
al historial.

In [18]:
# ==============================================================================
# CELDA 18: Conversación de seguimiento (usa la memoria)
# ==============================================================================
r1 = qa_mem.invoke({"question": "What is the smoking policy?"})
print("1:", r1["answer"], "\n")

r2 = qa_mem.invoke({"question": "List the points in it"})        # "it" = la política de fumar
print("2:", r2["answer"], "\n")

r3 = qa_mem.invoke({"question": "What is the aim of it?"})       # "it" sigue siendo la política de fumar
print("3:", r3["answer"])

1: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. Key points of the policy include:

- Smoking is only permitted in designated smoking areas, as marked by appropriate signage.
- Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited, including the use of electronic cigarettes and vaping devices.
- All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.
- Proper disposal of cigarette butts and related materials in designated receptacles is required; littering on company premises is prohibited.
- Smoking is not permitted in company vehicles.
- Non-compliance may lead to appropriate disciplinary action, which could include fines or, for employees, possible termination of employment.
- The policy will be reviewed periodically to ensure its alignment with evolving legal requirements and best practices. 

2: The key

**Resultado observado:** en las preguntas 2 y 3 el modelo interpretó "it" como *la
política de fumar*: enumeró sus puntos y explicó su objetivo. La memoria funciona.

### Celda 19: ¿Qué guardó la memoria?

In [19]:
# ==============================================================================
# CELDA 19: Inspeccionar el contenido de la memoria
# ==============================================================================
for m in memory.chat_memory.messages:
    print(type(m).__name__, ":", m.content[:100])   # HumanMessage / AIMessage, primeros 100 caracteres

HumanMessage : What is the smoking policy?
AIMessage : The Smoking Policy has been established to provide clear guidance and expectations concerning smokin
HumanMessage : List the points in it
AIMessage : The key points of the smoking policy include:

1. **Policy Purpose**: To provide clear guidance and 
HumanMessage : What is the aim of it?
AIMessage : The aim of the smoking policy is to provide clear guidance and expectations concerning smoking on co


**Resultado observado:** 6 mensajes (3 `HumanMessage` + 3 `AIMessage`), en orden. Cada
turno queda guardado como un par pregunta-respuesta.

## 7. Agente interactivo

### Celda 20: Definir el agente

Se junta todo (retriever + LLM + memoria) en una función con un bucle de preguntas y
respuestas. La memoria se crea **dentro** de la función, así que cada llamada a
`agente()` empieza una conversación nueva. Termina al escribir `quit`, `exit` o `bye`.

In [20]:
# ==============================================================================
# CELDA 20: Definición del agente conversacional
# ==============================================================================
def agente():
    # Memoria nueva en cada llamada: cada sesión empieza sin historial.
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer"
    )
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        chain_type="stuff",
        retriever=docsearch.as_retriever(),   # documento de políticas
        memory=memory,
        return_source_documents=False
    )

    while True:
        query = input("Question: ")           # en VS Code el cuadro de texto aparece ARRIBA, no en la celda

        if query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break

        result = chain.invoke({"question": query})
        print("Answer:", result["answer"], "\n")

### Celda 21: Ejecutar el agente

Preguntas de prueba, en este orden (las últimas dependen de la memoria):
`What is the smoking policy?` → `Can you list all points of it?` → `Can you summarize it?` → `quit`.

> ⚠️ Mientras el bucle está activo **no se pueden ejecutar otras celdas**. Para salir,
> escribe `quit` (o usa el botón de detener del notebook).
> Las preguntas escritas en el cuadro de entrada no se guardan en la salida; solo se
> ven las respuestas. Esta ejecución se cortó con el botón de detener; se omitió el
> traceback `KeyboardInterrupt` para no llenar el notebook de ruido.

In [21]:
# ==============================================================================
# CELDA 21: Ejecutar el agente (bucle interactivo)
# ==============================================================================
agente()

Answer: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. Key points of the policy include:

- Smoking is only permitted in designated smoking areas, as marked by appropriate signage.
- Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited, including the use of electronic cigarettes and vaping devices.
- All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.
- Proper disposal of cigarette butts and related materials in designated receptacles is required; littering on company premises is prohibited.
- Smoking is not permitted in company vehicles.
- Non-compliance may lead to appropriate disciplinary action, which could include fines or, for employees, possible termination of employment.
- The policy will be reviewed periodically to ensure its alignment with evolving legal requirements and best practices. 

Answe

## 8. Ejercicios

## Ejercicio 1: Trabajar con un documento propio

Se repite el pipeline con otro documento (`stateOfUnion.txt`, un discurso político en
inglés). Para **no mezclarlo** con las políticas de la empresa se usa una **colección de
Chroma distinta** (`collection_name="state_of_union"`).

### Celda 22: Descargar el segundo documento

In [22]:
# ==============================================================================
# CELDA 22 (Ejercicio 1): Descargar el segundo documento (solo si no existe)
# ==============================================================================
filename2 = 'stateOfUnion.txt'     # nombres nuevos (filename2/url2) para no pisar el documento de políticas
url2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'

if not os.path.exists(filename2):
    wget.download(url2, out=filename2)
    print('\nfile downloaded')
else:
    print('file already exists, skipping download')


file downloaded


### Celda 23: Inspeccionar el documento

Antes de procesar un documento conviene mirar su tamaño y su contenido.

In [23]:
# ==============================================================================
# CELDA 23 (Ejercicio 1): Inspección rápida del documento
# ==============================================================================
with open(filename2, 'r', encoding='utf-8') as f:
    contents2 = f.read()

print("Caracteres:", len(contents2))
print("Palabras aprox.:", len(contents2.split()))
print("\n--- Primeros 800 caracteres ---\n")
print(contents2[:800])

Caracteres: 38539
Palabras aprox.: 6469

--- Primeros 800 caracteres ---

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the U


**Resultado observado:** ~38 500 caracteres (~6 500 palabras): es un texto mucho más largo que
las políticas, así que aquí sí tiene sentido hacer retrieval.

### Celda 24: Dividir en chunks y comprobar el resultado

Además de dividir, se comprueba **cuántos saltos de línea dobles** hay (el splitter corta
ahí) y qué longitud tienen los chunks resultantes.

In [24]:
# ==============================================================================
# CELDA 24 (Ejercicio 1): Dividir en chunks y diagnosticar el corte
# ==============================================================================
loader2 = TextLoader(filename2, encoding="utf-8")
documents2 = loader2.load()

# Cuántos puntos de corte tiene el texto (el splitter corta en "\n\n").
texto2 = documents2[0].page_content
print("Saltos dobles (\\n\\n):", texto2.count("\n\n"))
print("Saltos simples (\\n):", texto2.count("\n"))

text_splitter2 = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts2 = text_splitter2.split_documents(documents2)

print("\nChunks:", len(texts2))
print("Longitud de los primeros 10:", [len(t.page_content) for t in texts2[:10]])
print("\n--- Chunk 1 ---\n")
print(texts2[0].page_content[:400])

Saltos dobles (\n\n): 358
Saltos simples (\n): 722

Chunks: 42
Longitud de los primeros 10: [939, 995, 959, 947, 976, 951, 954, 982, 893, 916]

--- Chunk 1 ---

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Con


**Resultado observado:** 358 saltos dobles → **42 chunks**. Los 10 primeros miden entre
893 y 995 caracteres y **no apareció ningún aviso** de chunk mayor a 1000 (a diferencia del
documento de políticas, donde 7 de 16 lo superaban): este texto tiene muchos puntos de corte.

### Celda 25: Embeddings y colección propia

In [25]:
# ==============================================================================
# CELDA 25 (Ejercicio 1): Embeddings + Chroma en una colección separada
# ==============================================================================
# Comprobación rápida antes de gastar tiempo en embeddings
n = len(texts2)
print("Chunks a procesar:", n)
if n < 10:
    print("⚠️ Muy pocos chunks: probablemente el splitter no encontró dónde cortar.")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

docsearch2 = Chroma.from_documents(
    texts2,
    embeddings,
    collection_name="state_of_union"   # colección separada de la de políticas
)
print("document ingested")

Chunks a procesar: 42
document ingested


### Celda 26: Probar la búsqueda **sin LLM**

Si un RAG responde mal, hay que saber si falló la **búsqueda** (trajo chunks
irrelevantes) o el **modelo** (tuvo el chunk correcto y respondió mal). Esta celda aísla
la primera parte.

In [26]:
# ==============================================================================
# CELDA 26 (Ejercicio 1): Búsqueda por similitud, sin LLM
# ==============================================================================
resultados = docsearch2.similarity_search("What does the speech say about the economy?", k=2)

for i, doc in enumerate(resultados, 1):
    print(f"--- Resultado {i} ---")
    print(doc.page_content[:300])
    print()

--- Resultado 1 ---
As Ohio Senator Sherrod Brown says, “It’s time to bury the label “Rust Belt.” 

It’s time. 

But with all the bright spots in our economy, record job growth and higher wages, too many families are struggling to keep up with the bills.  

Inflation is robbing them of the gains they might otherwise fe

--- Resultado 2 ---
We’re going after the criminals who stole billions in relief money meant for small businesses and millions of Americans.  

And tonight, I’m announcing that the Justice Department will name a chief prosecutor for pandemic fraud. 

By the end of this year, the deficit will be down to less than half w



**Resultado observado:** el resultado 1 trata de economía (empleo, salarios, inflación);
el 2 es cercano pero más tangencial (fraude pandémico y déficit). La búsqueda por
similitud devuelve lo *más parecido*, no siempre lo *perfecto*.

### Celda 27: Preguntar al discurso con el LLM

Ojo: se usa `docsearch2` (el discurso), no `docsearch` (las políticas).

In [27]:
# ==============================================================================
# CELDA 27 (Ejercicio 1): RAG sobre el segundo documento
# ==============================================================================
qa_sou = RetrievalQA.from_chain_type(
    llm=llm,                                  # qwen2.5:7b
    chain_type="stuff",                       # mete todos los chunks recuperados en un solo prompt
    retriever=docsearch2.as_retriever(),      # ojo: docsearch2, el del discurso
    chain_type_kwargs={"prompt": PROMPT},     # el prompt estricto ("no lo menciona")
    return_source_documents=True,
)

r = qa_sou.invoke("What does the speech say about the economy?")
print("Respuesta:\n", r["result"])
print("\nChunks usados:", len(r["source_documents"]))   # k=4 por defecto

Respuesta:
 The speech discusses several aspects of the economy, including record job growth and higher wages, but also mentions that too many families are struggling to keep up with bills due to inflation. It highlights the roaring back of the economy after the pandemic, issues with supply chains and factory production, and the impact on prices, particularly in the automotive industry. The speaker emphasizes the priority of getting prices under control and discusses strategies such as demanding more competition, using taxpayer dollars to support American jobs through "Buy American" policies, and passing legislation like the Bipartisan Innovation Act to invest in emerging technologies and manufacturing. Additionally, it contrasts past economic policies with current ones, suggesting that previous tax cuts for the top 1% did not benefit everyone equally.

Chunks usados: 4


### Celda 28: ¿La respuesta sale realmente de los chunks?

Que una respuesta suene bien no prueba que sea fiel. Se buscan las afirmaciones clave
de la respuesta dentro de los 4 chunks que recibió el modelo.

In [28]:
# ==============================================================================
# CELDA 28 (Ejercicio 1): Verificar que las afirmaciones están en las fuentes
# ==============================================================================
afirmaciones = [
    "record job growth",
    "supply chain",
    "Buy American",
    "Bipartisan Innovation Act",
    "top 1%",
]

fuentes = " ".join(d.page_content for d in r["source_documents"]).lower()

for a in afirmaciones:
    estado = "✅ está en las fuentes" if a.lower() in fuentes else "❌ NO está en los 4 chunks"
    print(f"{a:28} {estado}")

record job growth            ✅ está en las fuentes
supply chain                 ✅ está en las fuentes
Buy American                 ✅ está en las fuentes
Bipartisan Innovation Act    ✅ está en las fuentes
top 1%                       ✅ está en las fuentes


**Resultado observado:** las 5 afirmaciones aparecen en los chunks recuperados. Límite
de esta prueba: comprueba que la *frase* estaba en el contexto, no que el modelo la
haya *interpretado* bien.

### Celdas 29 y 30: Pregunta trampa sobre el discurso

Primero se verifica (con palabras completas) que el tema elegido **no** está en el
texto; después se pregunta. Una pregunta trampa solo vale si de verdad no hay respuesta.

In [29]:
# ==============================================================================
# CELDA 29 (Ejercicio 1): Confirmar que el discurso NO habla de criptomonedas
# ==============================================================================
import re

for palabra in ["cryptocurrency", "bitcoin", "crypto"]:
    n = len(re.findall(rf"\b{palabra}\b", contents2, re.IGNORECASE))
    print(f"'{palabra}': {n} coincidencias")

'cryptocurrency': 0 coincidencias
'bitcoin': 0 coincidencias
'crypto': 0 coincidencias


In [30]:
# ==============================================================================
# CELDA 30 (Ejercicio 1): Pregunta trampa sobre el discurso
# ==============================================================================
r2 = qa_sou.invoke("What does the speech say about cryptocurrency?")
print("Respuesta:\n", r2["result"])

Respuesta:
 The document does not mention this.


**Resultado observado:** 0 coincidencias para los tres términos, y el modelo respondió
`"The document does not mention this."`: con `qwen2.5:7b` y el prompt estricto respeta
el rechazo también en este segundo documento.

## Ejercicio 2: Devolver la fuente de cada respuesta

A veces no basta con la respuesta: se quiere ver **de qué parte del documento salió**.
Con `return_source_documents=True` el resultado es un diccionario con `result` (la
respuesta) y `source_documents` (los chunks que recibió el modelo, ordenados por
similitud).

### Celda 31: Respuesta + fuente principal

In [31]:
# ==============================================================================
# CELDA 31 (Ejercicio 2): Devolver la fuente de la respuesta
# ==============================================================================
qa_src = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),    # docsearch = documento de políticas
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True,          # <- clave del ejercicio
)

results = qa_src.invoke("Can I smoke in company vehicles?")

print("Respuesta:\n", results["result"])
print("\nFuente principal:\n")
print(results["source_documents"][0].page_content)   # [0] = el chunk más similar

Respuesta:
 No Smoking in Company Vehicles: Smoking is not permitted in company vehicles, whether they are owned or leased, to maintain the condition and cleanliness of these vehicles.

Fuente principal:

Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.
Designated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.
Smoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.
Compliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local 

**Resultado observado:** la frase de la respuesta aparece casi igual en la fuente. Esa
fuente es la política de fumar **completa en un solo chunk de 1678 caracteres** (por
encima de `chunk_size=1000`, porque la política no tiene líneas en blanco internas donde
cortar; ver Celda 4).

### Celda 32: Revisar las 4 fuentes, no solo la primera

El modelo recibe **4** chunks. Mirar solo el primero deja fuera a los otros tres.

In [32]:
# ==============================================================================
# CELDA 32 (Ejercicio 2): Inspeccionar todas las fuentes recuperadas
# ==============================================================================
for i, doc in enumerate(results["source_documents"], 1):
    contiene = "vehicles" in doc.page_content.lower()
    print(f"--- Fuente {i} ---")
    print("Metadatos:", doc.metadata)                  # con TextLoader solo trae el nombre del archivo
    print("¿Menciona 'vehicles'?:", contiene)
    print("Inicio:", doc.page_content[:120].replace("\n", " "), "...")
    print()

--- Fuente 1 ---
Metadatos: {'source': 'companyPolicies.txt'}
¿Menciona 'vehicles'?: True
Inicio: Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on ...

--- Fuente 2 ---
Metadatos: {'source': 'companyPolicies.txt'}
¿Menciona 'vehicles'?: False
Inicio: 5.	Smoking Policy ...

--- Fuente 3 ---
Metadatos: {'source': 'companyPolicies.txt'}
¿Menciona 'vehicles'?: False
Inicio: Policy Objective: The Drug and Alcohol Policy is established to establish clear expectations and guidelines for the resp ...

--- Fuente 4 ---
Metadatos: {'source': 'companyPolicies.txt'}
¿Menciona 'vehicles'?: False
Inicio: The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mob ...



**Resultado observado:**

| Fuente | Contenido | ¿Útil? |
|---|---|---|
| 1 | La política de fumar completa (menciona *vehicles*) | ✅ |
| 2 | Solo el título `5. Smoking Policy` | ➖ coincide en el tema pero no aporta contenido |
| 3 | Política de alcohol y drogas | ❌ relleno |
| 4 | Política de móviles | ❌ relleno |

El retriever **siempre devuelve k chunks**, sea cual sea su relevancia: el ranking puso
primero el chunk correcto, pero los otros tres son ruido que el modelo también recibe.

## Ejercicio 3: Usar otro modelo

El enunciado original pide cambiar a un modelo de watsonx (Mistral). Aquí no se usó
watsonx; la **idea** del ejercicio (comparar modelos con la misma prueba) se cubrió en la
**Sección 5** con `llama3.2` frente a `qwen2.5:7b`. Queda pendiente probar otros
(p. ej. `ollama pull mistral`) cambiando la variable `MODELO` de la Celda 15.

## 9. Conclusiones

### Lo que se logró
- Un pipeline RAG completo (**cargar → dividir → embeber → almacenar → recuperar → generar**)
  funcionando **100% en local** con Ollama, Chroma y LangChain, sin API keys.
- Los 4 objetivos del lab cubiertos, con los Ejercicios 1 y 2 completos y el 3 cubierto
  en su idea (comparación de modelos locales).
- Cada respuesta relevante se **contrastó con el texto fuente** en lugar de confiar en que
  "sonaba bien".

### Hallazgos (con la evidencia de este notebook)
1. **Sin contexto, el modelo responde con seguridad y se equivoca** (Celda 6: "RAG" como
   "Red de Ayuda y Rescate"). RAG reduce ese riesgo, pero no lo elimina.
2. **Con un modelo de 3B, el prompt no basta.** Ante la pregunta trampa, `llama3.2` respondió
   sobre un tema cercano (celdas 9 y 11) y, con prompt estricto y `temperature=0`, llegó a
   *inventar* que "incluyendo comer" estaba en la política (celda 12; comprobado con 0
   coincidencias en la celda 14).
3. **Cambiar el modelo sí ayudó**, con el resto igual: `qwen2.5:7b` respondió
   *"The document does not mention this."* en las políticas (celda 15) y en el discurso
   (celda 30), y siguió respondiendo con contenido a la pregunta de control.
4. **La recuperación es la otra mitad del sistema.** El retriever devuelve siempre k=4 chunks: en
   la celda 32 solo 1 de 4 era útil, uno era solo un título y dos eran de otras políticas. Poner
   primero el chunk correcto no impide que el modelo reciba ruido.
5. **El chunking depende de la estructura del texto, no de `chunk_size`.** Las políticas
   (bloques sin líneas en blanco) dieron 16 chunks, 7 de ellos de 1624 a 2032 caracteres;
   el discurso (358 saltos dobles) dio 42 chunks, sin avisos de tamaño excedido (los 10 primeros de 893 a 995).
6. **La memoria funciona**: `ConversationalRetrievalChain` + `ConversationBufferMemory`
   resolvió "it" en las preguntas de seguimiento (celda 18). Pero `ConversationBufferMemory`
   está **deprecada** (celda 17).
7. **Hay que verificar también la verificación.** `"eat" in texto` dio `True` por coincidir
   con subcadenas; con `\b` (palabras completas) dio 0 (celdas 13 y 14).

### Limitaciones (para no sobre-interpretar)
- Una sola pregunta trampa por documento y una corrida por configuración: **no es un benchmark**.
- En la celda 15 cambian a la vez el tamaño (3B → 7B) y la familia (Llama → Qwen):
  no se puede atribuir la mejora a solo uno de los dos factores.
- Los embeddings (`nomic-embed-text`) difieren de los del lab original, así que el ranking
  de chunks no es idéntico al de IBM.
- Documentos pequeños (16 y 42 chunks): el comportamiento con corpus grandes no se evaluó.
- Las celdas con `temperature=0.5` (6–11) pueden variar entre ejecuciones.
- El agente (y la cadena con memoria) usa los prompts **por defecto**, no el estricto; la
  pregunta trampa no se probó ahí.

### Siguientes pasos
- Sustituir `CharacterTextSplitter` por **`RecursiveCharacterTextSplitter`** con `chunk_overlap`
  (ya usado en el lab 02) y comparar número y tamaño de chunks.
- Probar **k más bajo** (`as_retriever(search_kwargs={"k": 2})`) o `search_type="mmr"` para reducir el ruido.
- Para resumir un documento **corto**, probar a pasar el texto completo al modelo en vez de hacer retrieval.
- Migrar la memoria a `create_agent` con *checkpointing*, como indica el aviso de deprecación (celda 17).
- Medir en serio: un conjunto pequeño de preguntas (con respuesta y sin ella), varias corridas,
  y calcular una tasa de aciertos por modelo.
- Probar más modelos locales y llevar el prompt estricto también al agente.

---

*Notebook adaptado del lab de IBM / Skills Network "Summarize Private Documents Using RAG,
LangChain, and LLMs" para correr 100% en local con Ollama. Las salidas corresponden a una
ejecución real en Windows con 16 GB de RAM y Python 3.12.*